In [ ]:
import pandas as pd
import numpy as np
import altair as alt

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold

### Add any other import statements you need below

In [28]:
cpi = pd.read_excel(r'cpi-u-2026\cpi-u-202609.xlsx',header=3)
cpi.iloc[0,0:3] = cpi.columns[:3]
cpi.columns = cpi.iloc[0]
cpi = cpi.iloc[3:]
unadjusted = cpi.iloc[:,[0,1]+list(range(3,16))]
unadjusted.head(10)

,Indent Level,Expenditure category,Sep.\n2025,Oct.\n2025,Nov.\n2025,Dec.\n2025,Jan.\n2026,Feb.\n2026,Mar.\n2026,Apr.\n2026,May\n2026,Jun.\n2026,Jul.\n2026,Aug.\n2026,Sep.\n2026
3,0.0,All items,324.8,–,324.122,324.054,325.252,326.785,330.213,333.02,335.123,333.952,333.918,334.98,336.94
4,1.0,Food,342.405,–,341.721,343.795,345.165,346.564,346.796,348.499,349.032,349.731,350.164,350.418,350.905
5,2.0,Food at home,316.042,–,313.954,315.664,317.431,318.898,318.755,320.859,321.047,321.631,321.643,321.5,321.758
6,3.0,Cereals and bakery products,361.894,–,361.255,361.984,366.83,367.174,365.909,365.366,367.3,368.8,369.882,369.171,369.585
7,4.0,Cereals and cereal products,283.727,–,287.875,286.9,291.766,290.796,289.157,289.54,288.902,291.89,293.913,291.389,292.743
8,5.0,Flour and prepared flour mixes,329.035,–,319.553,322.118,329.769,326.768,324.584,317.309,328.778,328.609,328.494,332.125,325.526
9,5.0,Breakfast cereal(4),277.496,–,284.674,281.923,287.902,285.565,284.357,284.219,282.677,285.006,287.933,283.314,285.768
10,5.0,"Rice, pasta, cornmeal",285.402,–,288.994,289.102,292.296,293.145,291.18,293.933,291.25,295.711,297.399,295.401,297.621
11,6.0,Rice(4)(5)(6),189.004,–,194.896,196.782,196.494,197.657,199.825,199.097,196.817,198.071,199.886,200.134,202.336
12,4.0,Bakery products(4),407.689,–,404.131,405.897,410.698,411.85,410.825,409.715,413.223,413.795,414.287,414.702,414.531


In [29]:
adjusted = cpi.iloc[:,[0,1]+list(range(16,29))]
adjusted.head(10)

,Indent Level,Expenditure category,Sep.\n2025,Oct.\n2025,Nov.\n2025,Dec.\n2025,Jan.\n2026,Feb.\n2026,Mar.\n2026,Apr.\n2026,May\n2026,Jun.\n2026,Jul.\n2026,Aug.\n2026,Sep.\n2026
3,0.0,All items,324.245,–,325.063,326.031,326.588,327.46,330.293,332.407,333.979,332.568,332.813,334.131,336.298
4,1.0,Food,342.086,–,342.386,344.632,345.271,346.622,346.603,348.349,348.892,349.609,349.881,350.309,350.477
5,2.0,Food at home,315.54,–,315,316.982,317.589,318.985,318.462,320.633,320.836,321.446,321.216,321.335,321.111
6,3.0,Cereals and bakery products,361.434,–,362.03,363.716,367.941,367.31,365.164,365.556,366.904,368.002,368.665,368.839,369.116
7,4.0,Cereals and cereal products,283.735,–,288.876,289.779,293.388,291.753,287.894,289.879,288.199,290.354,291.461,290.523,292.752
8,5.0,Flour and prepared flour mixes,326.425,–,329.407,331.574,328.973,327.027,323.23,318.929,327.064,323.568,322.818,327.343,322.942
9,5.0,Breakfast cereal(4),277.496,–,284.674,281.923,287.902,285.565,284.357,284.219,282.677,285.006,287.933,283.314,285.768
10,5.0,"Rice, pasta, cornmeal",286.22,–,290.249,290.597,293.281,292.633,289.922,293.622,290.118,294.726,295.484,295.006,298.475
11,6.0,Rice(4)(5)(6),189.004,–,194.896,196.782,196.494,197.657,199.825,199.097,196.817,198.071,199.886,200.134,202.336
12,4.0,Bakery products(4),407.689,–,404.131,405.897,410.698,411.85,410.825,409.715,413.223,413.795,414.287,414.702,414.531
